In [ ]:
# 環境準備
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("ライブラリのインポート: 完了")

In [ ]:
# ファイルパスの設定
from pathlib import Path

DATA_DIR = Path("/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra")

TRAIN_PATH = DATA_DIR / "train.parquet"
TEST_PATH = DATA_DIR / "test.parquet"

print(TRAIN_PATH.exists(), TEST_PATH.exists())
print("ファイルパスの設定: 完了")

In [ ]:
# 学習データの一部を読み込む
import pyarrow.parquet as pq

parquet = pq.ParquetFile(TRAIN_PATH)

train = next(parquet.iter_batches(batch_size=10000)).to_pandas() # 先頭10000スペクトルを読み込む

print("スペクトル数:", len(train))
display(train.head())

In [ ]:
# データ構造の確認
print("データ形状:", train.shape)
print("\nカラム一覧:")
print(train.columns.tolist())

print("\nデータ数:")
print(train.dtypes)

In [ ]:
# 分子構造とスペクトル数の確認
print("スペクトル数:", len(train))
print("分子構造数:", train["inchikey14"].nunique())

display(train["inchikey14"].value_counts().head(10))

In [ ]:
# 測定条件の確認
display(train["ionization_mode"].value_counts())
display(train["adduct"].value_counts().head(10))
display(train["instrument_type"].value_counts().head(10))

In [ ]:
# スペクトルのピークデータを確認
sample = train.iloc[0]

mzs = sample["ms2_mzs"]
intensities = sample["ms2_normalized_intensities"]

print("SMILES:", sample["normalized_smiles"])
print("Precursor m/z:", sample["precursor_mz"])
print("ピーク数:", len(mzs))

display(pd.DataFrame({
    "m/z": mzs,
    "Intensity": intensities
}).head(10))

In [ ]:
# MS/MSスペクトルの可視化
plt.figure(figsize=(12, 4))

plt.vlines(mzs, 0, intensities)

plt.xlabel("m/z")
plt.ylabel("Normalized Intensity")
plt.title("MS/MS Spectrum")
plt.show()

In [ ]:
# テストデータの確認
test = pd.read_parquet(TEST_PATH)

print("テストデータ形状:", test.shape)
print("分子数:", test["molecule_id"].nunique())

display(test.head())

In [ ]:
# 分子ごとのスペクトル数を確認
spectra_per_molecule = test.groupby("molecule_id").size()

print(spectra_per_molecule.describe())

plt.hist(spectra_per_molecule, bins=15)
plt.xlabel("Number of Spectra")
plt.ylabel("Number of Molecules")
plt.show()

In [ ]:
# テストデータの質量を準備
test = test.reset_index(drop=True)

test_masses = {
    adduct: np.sort(group["precursor_mz"].dropna().to_numpy())
    for adduct, group in test.groupby("adduct")
}

MASS_PPM = 35

# 全学習データから候補を抽出

columns = [
    "precursor_mz", "adduct", "normalized_smiles",
    "inchikey14", "ms2_mzs", "ms2_normalized_intensities"
]

chunks = []

for batch in parquet.iter_batches(batch_size=50000, columns=columns):
    df = batch.to_pandas()
    selected = np.zeros(len(df), dtype=bool)

    for adduct, masses in test_masses.items():
        if len(masses) == 0:
            continue

        mask = (df["adduct"] == adduct).to_numpy()
        mz = df.loc[mask, "precursor_mz"].to_numpy()
        pos = np.searchsorted(masses, mz)

        left = masses[np.clip(pos - 1, 0, len(masses) - 1)]
        right = masses[np.clip(pos, 0, len(masses) - 1)]
        ppm = np.minimum(abs(mz - left), abs(mz - right)) / mz * 1e6

        selected[np.flatnonzero(mask)] = ppm <= MASS_PPM

    chunks.append(df.loc[selected])

train = pd.concat(chunks, ignore_index=True)

print("参照スペクトル:", len(train))
print("参照分子構造:", train["inchikey14"].nunique())

In [ ]:
# ビニングのパラメータ設定
BIN_WIDTH = 0.1
MAX_MZ = 1250

N_BINS = int(MAX_MZ / BIN_WIDTH)

print("ビン数:", N_BINS)

In [ ]:
# 可変長スペクトルを固定長に変換する
from scipy.sparse import csr_matrix

def bin_spectrum(mzs, intensities):
    mzs = np.asarray(mzs)
    intensities = np.asarray(intensities)

    valid = (mzs >= 40) & (mzs < MAX_MZ) & (intensities >= 0.01)
    mzs, intensities = mzs[valid], intensities[valid]

    top = np.argsort(intensities)[-100:]
    mzs, intensities = mzs[top], intensities[top]

    indices = (mzs / BIN_WIDTH).astype(int)
    weights = np.sqrt(intensities) * np.sqrt(mzs)

    vector = np.zeros(N_BINS, dtype=np.float32)
    np.maximum.at(vector, indices, weights)

    return csr_matrix(vector.reshape(1, -1))

In [ ]:
# ビニング後のベクトルをL2正規化する
from sklearn.preprocessing import normalize

def normalize_spectrum(vector):
    return normalize(vector, norm="l2")

In [ ]:
# 特徴量の生成
from scipy.sparse import vstack

X_train = vstack([
    normalize_spectrum(bin_spectrum(mzs, intensities))
    for mzs, intensities in zip(
        train["ms2_mzs"],
        train["ms2_normalized_intensities"]
    )
]).tocsr()

print("特徴量:", X_train.shape)

In [ ]:
# 前処理結果の確認
vector = X_train[0].toarray().ravel()

print("元のピーク数:", len(train.iloc[0]["ms2_mzs"]))
print("変換後の次元数:", len(vector))
print("L2ノルム:", np.linalg.norm(vector))

plt.figure(figsize=(12, 4))
plt.plot(np.arange(N_BINS) * BIN_WIDTH, vector)
plt.xlabel("m/z")
plt.ylabel("Normalized Intensity")
plt.title("Binned MS/MS Spectrum")
plt.show()

In [ ]:
# 参照データの準備
reference = train[[
    "precursor_mz",
    "adduct",
    "normalized_smiles"
]].reset_index(drop=True)

print("参照スペクトル数:", len(reference))
print("特徴量の形状:", X_train.shape)

display(reference.head())

In [ ]:
# 候補スペクトルの絞り込み
def select_candidates(precursor_mz, adduct, reference):
    ppm = (
        (reference["precursor_mz"] - precursor_mz).abs()
        / precursor_mz * 1e6
    )

    mask = (reference["adduct"] == adduct) & (ppm <= MASS_PPM)

    return np.flatnonzero(mask.to_numpy())

In [ ]:
# 分子構造のランキング
def rank_molecules(results, top_k=25):
    ranked = results.drop_duplicates(
        subset="normalized_smiles"
    )

    return ranked.head(top_k)[["normalized_smiles", "similarity"]]

print("分子構造のランキング: 完了")

In [ ]:
# コサイン類似度による検索
def search_spectra(query_vector, precursor_mz, adduct,
                   X_reference, reference):

    indices = select_candidates(precursor_mz, adduct, reference)

    scores = (
        X_reference[indices] @ query_vector.T
    ).toarray().ravel()

    results = reference.iloc[indices].copy()
    results["similarity"] = scores

    return results.sort_values("similarity", ascending=False)

print("コサイン類似度による検索: 完了")

In [ ]:
# 検索モデルの動作確認
sample = train.iloc[0]
query_vector = X_train.getrow(0)

results = search_spectra(
    query_vector,
    sample["precursor_mz"],
    sample["adduct"],
    X_train,
    reference
)

ranked = rank_molecules(results)

display(ranked)
print("検索モデルの動作確認: 完了")

In [ ]:
from scipy.sparse import issparse

print(issparse(X_train))
print(X_train.shape)

In [ ]:
# 検証対象の分子を選択
counts = train["inchikey14"].value_counts()

valid_keys = counts[counts >= 2].index
valid_data = train[train["inchikey14"].isin(valid_keys)]

print("検証可能な分子数:", len(valid_keys))
print("対象スペクトル数:", len(valid_data))

In [ ]:
# 参照・検証データの分割
val_indices = (
    valid_data.groupby("inchikey14")
    .sample(n=1, random_state=42)
    .index[:100]
)

ref_indices = train.index.difference(val_indices)

train_ref = train.loc[ref_indices].reset_index(drop=True)
train_val = train.loc[val_indices].reset_index(drop=True)

print("参照スペクトル数:", len(train_ref))
print("検証スペクトル数:", len(train_val))

In [ ]:
# 検証用の検索モデルを準備
X_ref = X_train[ref_indices.to_numpy()]
X_val = X_train[val_indices.to_numpy()]

reference_val = train_ref[[
    "precursor_mz",
    "adduct",
    "normalized_smiles",
    "inchikey14"
]]

print("参照特徴量:", X_ref.shape)
print("検証特徴量:", X_val.shape)

In [ ]:
# 検証データの予測
predictions = []

for i, sample in train_val.iterrows():
    results = search_spectra(
        X_val[i],
        sample["precursor_mz"],
        sample["adduct"],
        X_ref,
        reference_val
    )

    ranked = results.drop_duplicates("inchikey14").head(25)
    predictions.append(ranked["inchikey14"].tolist())

print("検証データの予測: 完了")

In [ ]:
# MRR@25の計算

# 評価関数
def reciprocal_rank(predicted, actual):
    if actual in predicted:
        return 1 / (predicted.index(actual) + 1)

    return 0.0

# 全体のMRR@25
actuals = train_val["inchikey14"].tolist()

scores = [
    reciprocal_rank(pred, actual)
    for pred, actual in zip(predictions, actuals)
]

print("MRR@25:", np.mean(scores))

In [ ]:
# 予測結果の確認
evaluation = pd.DataFrame({
    "actual": actuals,
    "predicted_top1": [p[0] if p else None for p in predictions],
    "score": scores
})

display(evaluation.head(10))


top1_accuracy = np.mean(evaluation["score"] == 1.0)
top25_accuracy = np.mean(evaluation["score"] > 0)

print(f"Top-1 Accuracy: {top1_accuracy:.4f}")
print(f"Top-25 Accuracy: {top25_accuracy:.4f}")

In [ ]:
# テストデータの前処理
X_test = vstack([
    normalize_spectrum(bin_spectrum(mzs, intensities))
    for mzs, intensities in zip(
        test["ms2_mzs"],
        test["ms2_normalized_intensities"]
    )
]).tocsr()

print("テスト特徴量:", X_test.shape)

In [ ]:
# スペクトルごとの検索
def predict_spectrum(vector, sample):
    return search_spectra(
        vector,
        sample["precursor_mz"],
        sample["adduct"],
        X_train,
        reference
    )

In [ ]:
# 分子単位のランキング
def predict_molecule(group):
    results = []

    for idx, sample in group.iterrows():
        ranked = predict_spectrum(X_test[idx], sample)
        results.append(ranked)

    combined = pd.concat(results)

    if combined.empty:
        return []

    scores = (
        combined.groupby("normalized_smiles")["similarity"]
        .max()
        .sort_values(ascending=False)
    )

    return scores.head(25).index.tolist()

In [ ]:
# 全テストデータへの推論
test = test.reset_index(drop=True)

predictions_test = {}

for molecule_id, group in test.groupby("molecule_id"):
    predictions_test[molecule_id] = predict_molecule(group)

print("予測した分子数:", len(predictions_test))

In [ ]:
# 候補不足への対応
fallback = reference["normalized_smiles"].dropna().drop_duplicates().tolist()

for molecule_id, smiles in predictions_test.items():
    if not smiles:
        predictions_test[molecule_id] = fallback[:25]

In [ ]:
# 提出用DataFrameの作成

submission = pd.DataFrame({
    "molecule_id": predictions_test.keys(),
    "smiles": [
        ";".join(smiles)
        for smiles in predictions_test.values()
    ]
})

display(submission.head())

In [ ]:
# 提出形式の確認
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

assert set(submission["molecule_id"]) == set(sample_submission["molecule_id"])
assert submission["molecule_id"].is_unique

assert submission["smiles"].notna().all()
assert submission["smiles"].str.len().gt(0).all()
print("提出対象:", len(submission))
print("空の予測:", (submission["smiles"] == "").sum())


print("提出形式のチェック完了")

In [ ]:
# CSVファイルの保存

submission = sample_submission[["molecule_id"]].merge(
    submission,
    on="molecule_id",
    how="left"
)

submission.to_csv(
    "/kaggle/working/submission.csv",
    index=False
)

print("submission.csv を保存しました")
display(submission.head())